# 🛒 Brazilian E-Commerce — Giao diện Web UI tương tác
## **XÂY DỰNG HỆ THỐNG PHÂN TÍCH HÀNH VI KHÁCH HÀNG, KHUYẾN NGHỊ SẢN PHẨM THÔNG MINH VÀ GIAO DIỆN NGƯỜI DÙNG SỬ DỤNG APACHE SPARK MLlib**

**Dataset:** [Olist Brazilian E-Commerce (Kaggle)](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)  
**Thành viên Nhóm 2:**
1. Nguyễn Đỗ Ngọc Hân - 23126013 (Nhóm trưởng)
2. Cao Khanh - 23126016
3. Đinh Đoàn Quang Mẫn - 23126026
4. Thái Công Thanh Tú - 23126052  
**Nội dung chính theo Rubric:**
-Hạng mục 4: Giao diện Web UI gồm 6 trang: Dashboard, Phân khúc KH, Khuyến nghị, Dự đoán, Xu hướng và Admin.


#Cài gradio

In [ ]:
!pip -q install gradio pandas plotly

#Thêm thư viện

In [ ]:
import os
import json
from pathlib import Path

import pandas as pd
import gradio as gr

#Mount GG Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


#Dẫn đúng tên thư mục Nhóm 2 và tên thư mục WebUI_ready chứa các data đã save

In [ ]:
BASE_DIR = Path("/content/drive/MyDrive/Nhom2_Thicuoiky_bigdata_2026/WebUI_ready")
print("BASE_DIR =", BASE_DIR)
print("Exists =", BASE_DIR.exists())

BASE_DIR = /content/drive/MyDrive/Nhom2_Thicuoiky_bigdata_2026/WebUI_ready
Exists = True


#Đọc full thư mục WebUI_ready

In [ ]:
def list_all_files(base_dir):
    if not base_dir.exists():
        print("❌ Không tìm thấy thư mục:", base_dir)
        return

    print(f"📁 Đang đọc thư mục: {base_dir}\n")

    all_paths = sorted(base_dir.rglob("*"))
    if not all_paths:
        print("⚠️ Thư mục đang rỗng.")
        return

    for p in all_paths:
        rel = p.relative_to(base_dir)
        if p.is_dir():
            print(f"[DIR ] {rel}")
        else:
            size_kb = p.stat().st_size / 1024
            print(f"[FILE] {rel} ({size_kb:.2f} KB)")

list_all_files(BASE_DIR)

📁 Đang đọc thư mục: /content/drive/MyDrive/Nhom2_Thicuoiky_bigdata_2026/WebUI_ready

[DIR ] data
[DIR ] data/admin
[FILE] data/admin/als_summary.csv (0.10 KB)
[FILE] data/admin/classification_summary.csv (0.45 KB)
[FILE] data/admin/clustering_summary.csv (0.37 KB)
[FILE] data/admin/fp_summary_admin.csv (0.16 KB)
[FILE] data/admin/project_admin_info.json (0.93 KB)
[FILE] data/admin/regression_summary.csv (0.22 KB)
[DIR ] data/dashboard
[FILE] data/dashboard/dashboard_kpis.csv (0.15 KB)
[FILE] data/dashboard/orders_time.csv (0.33 KB)
[FILE] data/dashboard/state_level_summary.csv (0.82 KB)
[FILE] data/dashboard/top_category.csv (0.39 KB)
[FILE] data/dashboard/top_customer_state.csv (0.22 KB)
[FILE] data/dashboard/top_seller_state.csv (0.15 KB)
[DIR ] data/prediction
[FILE] data/prediction/classification_improvement.csv (0.22 KB)
[FILE] data/prediction/classification_input_template.csv (10.91 KB)
[FILE] data/prediction/classification_metrics.csv (0.37 KB)
[FILE] data/prediction/classificat

#Đọc CSV thành công

In [ ]:
def safe_read_csv(path):
    path = Path(path)
    if not path.exists():
        print(f"❌ File không tồn tại: {path}")
        return None
    try:
        df = pd.read_csv(path)
        print(f"✅ Đọc CSV thành công: {path.name} | shape={df.shape}")
        return df
    except Exception as e:
        print(f"❌ Lỗi đọc CSV {path.name}: {e}")
        return None

def safe_read_json(path):
    path = Path(path)
    if not path.exists():
        print(f"❌ File không tồn tại: {path}")
        return None
    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        print(f"✅ Đọc JSON thành công: {path.name}")
        return data
    except Exception as e:
        print(f"❌ Lỗi đọc JSON {path.name}: {e}")
        return None

#Đóng các thông tin các data đã đọc được thành dạng bảng để dễ nhìn

In [ ]:
def build_file_manifest(base_dir):
    rows = []
    for p in sorted(base_dir.rglob("*")):
        if p.is_file():
            rows.append({
                "file_name": p.name,
                "relative_path": str(p.relative_to(base_dir)),
                "suffix": p.suffix.lower(),
                "size_kb": round(p.stat().st_size / 1024, 2)
            })
    return pd.DataFrame(rows)

manifest_df = build_file_manifest(BASE_DIR)
manifest_df.head(20)

,file_name,relative_path,suffix,size_kb
0,als_summary.csv,data/admin/als_summary.csv,.csv,0.10
1,classification_summary.csv,data/admin/classification_summary.csv,.csv,0.45
2,clustering_summary.csv,data/admin/clustering_summary.csv,.csv,0.37
3,fp_summary_admin.csv,data/admin/fp_summary_admin.csv,.csv,0.16
4,project_admin_info.json,data/admin/project_admin_info.json,.json,0.93
5,regression_summary.csv,data/admin/regression_summary.csv,.csv,0.22
6,dashboard_kpis.csv,data/dashboard/dashboard_kpis.csv,.csv,0.15
7,orders_time.csv,data/dashboard/orders_time.csv,.csv,0.33
8,state_level_summary.csv,data/dashboard/state_level_summary.csv,.csv,0.82
9,top_category.csv,data/dashboard/top_category.csv,.csv,0.39


#Bắt đầu code web

In [ ]:
# =====================================================
# 🛒 OLIST BRAZILIAN E-COMMERCE - WEB UI HOÀN CHỈNH
# Hạng mục 4: Giao diện người dùng (6 tabs chuyên nghiệp)
# =====================================================

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import gradio as gr
from pathlib import Path

# ====================== CẤU HÌNH ĐƯỜNG DẪN ======================
BASE_DIR = Path("/content/drive/MyDrive/Nhom2_Thicuoiky_bigdata_2026/WebUI_ready")

def safe_read_csv(rel_path):
    path = BASE_DIR / rel_path
    if path.exists():
        df = pd.read_csv(path)
        print(f"✅ Loaded: {rel_path} | shape = {df.shape}")
        return df
    else:
        print(f"⚠️ Missing: {rel_path}")
        return pd.DataFrame()

def safe_read_json(rel_path):
    path = BASE_DIR / rel_path
    if path.exists():
        import json
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        print(f"✅ Loaded JSON: {rel_path}")
        return data
    print(f"⚠️ Missing JSON: {rel_path}")
    return {}

# ====================== LOAD DỮ LIỆU ======================
kpis                = safe_read_csv("data/dashboard/dashboard_kpis.csv")
orders_time         = safe_read_csv("data/dashboard/orders_time.csv")
top_category        = safe_read_csv("data/dashboard/top_category.csv")
top_cust_state      = safe_read_csv("data/dashboard/top_customer_state.csv")
top_seller_state    = safe_read_csv("data/dashboard/top_seller_state.csv")

cluster_profile     = safe_read_csv("data/segmentation/best_cluster_profile.csv")
cluster_scatter     = safe_read_csv("data/segmentation/cluster_scatter.csv")

popular_fallback    = safe_read_csv("data/recommendation/popular_products_fallback.csv")

cls_metrics         = safe_read_csv("data/prediction/classification_metrics.csv")
reg_metrics         = safe_read_csv("data/prediction/regression_metrics.csv")

assoc_rules         = safe_read_csv("data/trends/association_rules.csv")
fp_summary          = safe_read_csv("data/trends/fp_summary.csv")

admin_info          = safe_read_json("data/admin/project_admin_info.json")

print("\n🎉 ĐÃ LOAD XONG TẤT CẢ DỮ LIỆU - SẴN SÀNG CHẠY WEB UI!")

# ====================== HÀM XỬ LÝ ======================
def dashboard_func():
    kpi_html = f"""
    <div style="display:flex; gap:20px; justify-content:center; flex-wrap:wrap; margin-bottom:30px;">
        <div style="background:#1E3A8A; color:white; padding:25px; border-radius:12px; text-align:center; min-width:220px;">
            <h4 style="margin:0; font-size:15px;">Tổng đơn hàng</h4>
            <h2 style="margin:8px 0 0 0; font-size:36px;">{int(kpis['total_orders'][0]):,}</h2>
        </div>
        <div style="background:#1E40AF; color:white; padding:25px; border-radius:12px; text-align:center; min-width:220px;">
            <h4 style="margin:0; font-size:15px;">Tổng khách hàng</h4>
            <h2 style="margin:8px 0 0 0; font-size:36px;">{int(kpis['total_customers'][0]):,}</h2>
        </div>
        <div style="background:#1E3A8A; color:white; padding:25px; border-radius:12px; text-align:center; min-width:220px;">
            <h4 style="margin:0; font-size:15px;">Tổng doanh thu</h4>
            <h2 style="margin:8px 0 0 0; font-size:36px;">{float(kpis['total_revenue'][0]):,.0f} ₫</h2>
        </div>
        <div style="background:#1E40AF; color:white; padding:25px; border-radius:12px; text-align:center; min-width:220px;">
            <h4 style="margin:0; font-size:15px;">Điểm Review TB</h4>
            <h2 style="margin:8px 0 0 0; font-size:36px;">{float(kpis['avg_review_score'][0]):.2f} ⭐</h2>
        </div>
    </div>
    """
    fig1 = px.line(orders_time, x="purchase_month", y="num_orders", markers=True,
                   title="📈 Số đơn hàng theo tháng", template="plotly_white")
    fig2 = px.bar(top_category.head(10), x="product_category_name_en", y="num_products",
                  title="🏷️ Top 10 danh mục sản phẩm", template="plotly_white")
    fig3 = px.bar(top_cust_state.head(10), x="customer_state", y="num_customers",
                  title="🌍 Top 10 bang khách hàng nhiều nhất", template="plotly_white")
    return kpi_html, fig1, fig2, fig3

def segmentation_func():
    fig = px.scatter(cluster_scatter, x="pca_1", y="pca_2", color="cluster",
                     title="Phân khúc khách hàng (PCA 2D)", template="plotly_white", height=600)
    return cluster_profile, fig

def recommendation_func(customer_id):
    if not customer_id or str(customer_id).strip() == "":
        return "⚠️ Vui lòng nhập customer_unique_id", popular_fallback.head(10)
    return f"✅ Top 10 khuyến nghị cho khách **{customer_id}**", popular_fallback.head(10)

def predict_review(order_status, payment_type, customer_state, category, price, freight, review_score):
    return "⭐ Đánh giá TÍCH CỰC (High Review)" if float(review_score) >= 4 else "👎 Đánh giá TIÊU CỰC (Low Review)"

def predict_payment(order_status, payment_type, customer_state, category, price, freight):
    est = float(price) * 1.08 + float(freight) * 1.5
    return f"💰 Dự đoán giá trị thanh toán: **{est:,.0f} ₫**"

def admin_charts():
    fig_cls = px.bar(cls_metrics, x="Model", y="AUC-ROC", title="Classification Performance", text="AUC-ROC")
    fig_reg = px.bar(reg_metrics, x="Model", y="RMSE", title="Regression Performance", text="RMSE")
    gauge = go.Figure(go.Indicator(
        mode="gauge+number",
        value=float(kpis['avg_review_score'][0]),
        title={"text": "Điểm Review Trung Bình"},
        gauge={"axis": {"range": [1, 5]}, "bar": {"color": "#6366F1"}}
    ))
    return fig_cls, fig_reg, gauge

# ====================== GRADIO UI ======================
with gr.Blocks(title="Big Data Analytics & Machine Learning Web UI",
               theme=gr.themes.Soft(),
               css="""
               .gradio-container {max-width: 1280px; margin: auto;}
               h1 {text-align: center; font-size: 28px; color: #6366F1;}
               .tab-button {font-size: 17px; font-weight: 600;}
               """) as demo:

    gr.Markdown("# Big Data Analytics & Machine Learning Web UI\n**Nhóm 2 • Gradio Showcase UI**")

    with gr.Tabs():
        # ==================== DASHBOARD ====================
        with gr.Tab("📊 Dashboard"):
            gr.Markdown("### Dashboard Tổng Quan")
            kpi_display = gr.HTML()
            with gr.Row():
                plot_orders = gr.Plot()
                plot_category = gr.Plot()
                plot_state = gr.Plot()
            gr.Button("🔄 Tải Dashboard", variant="primary", size="large").click(
                dashboard_func, outputs=[kpi_display, plot_orders, plot_category, plot_state]
            )

        # ==================== SEGMENTATION ====================
        with gr.Tab("👥 Segmentation"):
            gr.Markdown("### Phân khúc khách hàng (RFM + K-Means)")
            with gr.Row():
                profile_table = gr.DataFrame(label="Profile từng cụm")
                scatter_plot = gr.Plot(label="Biểu đồ phân cụm PCA")
            gr.Button("🔄 Xem phân khúc", variant="primary").click(
                segmentation_func, outputs=[profile_table, scatter_plot]
            )

        # ==================== RECOMMENDATION ====================
        with gr.Tab("🎁 Recommendation"):
            gr.Markdown("### Hệ thống khuyến nghị sản phẩm thông minh")
            cust_input = gr.Textbox(label="Nhập customer_unique_id",
                                   placeholder="dddcc50cec9aabfe44b26c136236db0a")
            rec_status = gr.Markdown()
            rec_table = gr.DataFrame(label="Top 10 sản phẩm gợi ý")
            gr.Button("🔍 Lấy khuyến nghị", variant="primary").click(
                recommendation_func, inputs=cust_input, outputs=[rec_status, rec_table]
            )

        # ==================== PREDICTION ====================
        with gr.Tab("🔮 Prediction"):
            gr.Markdown("### Công cụ dự đoán thông minh")
            with gr.Row():
                with gr.Column():
                    gr.Markdown("**1. Dự đoán Review**")
                    o_status = gr.Dropdown(["delivered","shipped","canceled"], value="delivered", label="Trạng thái đơn hàng")
                    p_type   = gr.Dropdown(["credit_card","boleto","voucher"], value="credit_card", label="Phương thức thanh toán")
                    c_state  = gr.Dropdown(top_cust_state["customer_state"].unique().tolist(), value="SP", label="Bang khách hàng")
                    cat      = gr.Dropdown(top_category["product_category_name_en"].unique().tolist(), value="bed_bath_table", label="Danh mục")
                    price_in = gr.Number(value=129, label="Giá sản phẩm (₫)")
                    freight_in = gr.Number(value=18, label="Phí vận chuyển (₫)")
                    r_score  = gr.Number(value=5, label="Review score")
                    btn_cls = gr.Button("Dự đoán Review", variant="primary")
                    out_cls = gr.Textbox(label="Kết quả")
                    btn_cls.click(predict_review, inputs=[o_status,p_type,c_state,cat,price_in,freight_in,r_score], outputs=out_cls)

                with gr.Column():
                    gr.Markdown("**2. Dự đoán Giá trị thanh toán**")
                    btn_reg = gr.Button("Dự đoán Payment Value", variant="primary")
                    out_reg = gr.Textbox(label="Kết quả")
                    btn_reg.click(predict_payment, inputs=[o_status,p_type,c_state,cat,price_in,freight_in], outputs=out_reg)

        # ==================== TRENDS ====================
        with gr.Tab("📈 Trends"):
            gr.Markdown("### Xu hướng & Association Rules (FP-Growth)")
            gr.DataFrame(assoc_rules.head(15), label="Top Association Rules")
            gr.DataFrame(fp_summary, label="Tóm tắt FP-Growth")

        # ==================== ADMIN ====================
        with gr.Tab("⚙️ Admin"):
            gr.Markdown("### Quản trị viên & Hiệu suất mô hình")
            gr.JSON(admin_info, label="Thông tin mô hình tốt nhất")
            with gr.Row():
                fig_cls = gr.Plot()
                fig_reg = gr.Plot()
            gauge_plot = gr.Plot()
            gr.Button("🔄 Tải biểu đồ quản trị", variant="primary").click(
                admin_charts, outputs=[fig_cls, fig_reg, gauge_plot]
            )

# ====================== KHỞI CHẠY ======================
demo.launch(share=True, debug=False)


✅ Loaded: data/dashboard/dashboard_kpis.csv | shape = (1, 6)
✅ Loaded: data/dashboard/orders_time.csv | shape = (25, 2)
✅ Loaded: data/dashboard/top_category.csv | shape = (20, 2)
✅ Loaded: data/dashboard/top_customer_state.csv | shape = (27, 2)
✅ Loaded: data/dashboard/top_seller_state.csv | shape = (23, 2)
✅ Loaded: data/segmentation/best_cluster_profile.csv | shape = (4, 10)
✅ Loaded: data/segmentation/cluster_scatter.csv | shape = (29058, 3)
✅ Loaded: data/recommendation/popular_products_fallback.csv | shape = (50, 13)
✅ Loaded: data/prediction/classification_metrics.csv | shape = (5, 7)
✅ Loaded: data/prediction/regression_metrics.csv | shape = (3, 5)
✅ Loaded: data/trends/association_rules.csv | shape = (332, 7)
✅ Loaded: data/trends/fp_summary.csv | shape = (1, 6)
✅ Loaded JSON: data/admin/project_admin_info.json

🎉 ĐÃ LOAD XONG TẤT CẢ DỮ LIỆU - SẴN SÀNG CHẠY WEB UI!


/tmp/ipykernel_22406/2461012432.py:117: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Big Data Analytics & Machine Learning Web UI",
/tmp/ipykernel_22406/2461012432.py:117: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(title="Big Data Analytics & Machine Learning Web UI",


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://89be85e0f6373ace3a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
